In [ ]:
!nvidia-smi

Thu Apr 23 16:45:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             33W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd
from datasets import load_dataset

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("yavuzyilmaz/cosmetic-ingredients")
df = pd.DataFrame(ds['train'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
df = df.fillna("")
print(df.head())
print(df.describe())
print(df.columns)

          ingredient                                        description
0           Glycerin  Glycerin doesn’t sound very glamorous but it i...
1    Butylene Glycol  Butylene glycol, or let’s just call it BG, is ...
2           Squalene  Squalene is an oily liquid that originally com...
3       Ceteareth-20  A common functional ingredient that helps to k...
4  Glyceryl Stearate  A super common, waxy, white, solid stuff that ...
       ingredient                                        description
count        1020                                               1020
unique       1013                                               1016
top      No Title  A pre-neutralised form of super common thicken...
freq            8                                                  2
Index(['ingredient', 'description'], dtype='object')


In [ ]:
!pip install transformers


In [ ]:
df['full_info'] = "Ingredient: " + df['ingredient'] + " | Description: " + df['description']
documents = df['full_info'].tolist()

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoTokenizer
import transformers


max_words = max(len(doc.split()) for doc in documents)
avg_words = sum(len(doc.split()) for doc in documents) / len(documents)

print(f"Average word count: {avg_words:.2f}")
print(f"Longest document (words): {max_words}")

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

token_counts = [len(tokenizer.encode(doc)) for doc in documents]
max_tokens = max(token_counts)

print(f"Longest document (tokens): {max_tokens}")

model_max_length = tokenizer.model_max_length
print(f"Model max length: {model_max_length}")

Average word count: 111.45
Longest document (words): 1441
Longest document (tokens): 1945
Model max length: 131072


In [ ]:
!pip install rank_bm25 faiss-cpu

In [ ]:
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")

# --- Variant B: BM25 (Sparse Retrieval) ---
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

# --- Variant C: FAISS (Dense Retrieval) ---
embed_model = SentenceTransformer('microsoft/harrier-oss-v1-0.6b')
embeddings = embed_model.encode(documents, batch_size=4, show_progress_bar=True, device=device)

# Vector index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))


Device used: cuda


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
import os
import joblib

drive.mount('/content/drive')
path = "/content/drive/MyDrive/Studia_MJ/LLM_projekt"

if not os.path.exists(path):
    os.makedirs(path)

data_to_save = {
    'documents': documents,
    'bm25': bm25
}
joblib.dump(data_to_save, f'{path}/data_pack.joblib')


faiss.write_index(index, f"{path}/faiss_index.bin")

print(f"Everything saved to : {path}")
print("- Documents and BM25: data_pack.joblib")
print("- Index FAISS: faiss_index.bin")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Everything saved to : /content/drive/MyDrive/Studia_MJ/LLM_projekt
- Documents and BM25: data_pack.joblib
- Index FAISS: faiss_index.bin
